# Chào mừng đến với tuần học về RAG!!

## Chuyên viên tri thức

### Trợ lý hỏi đáp đóng vai trò là một chuyên viên tri thức
### Dành cho nhân viên của Insurellm, một công ty công nghệ bảo hiểm
### Trợ lý AI cần đưa ra câu trả lời chính xác và giải pháp phải có chi phí thấp.

Dự án này sẽ sử dụng RAG (Retrieval Augmented Generation — Sinh tăng cường truy xuất) để đảm bảo trợ lý hỏi đáp của chúng ta có độ chính xác cao.

Phiên bản triển khai đầu tiên này sẽ sử dụng một kiểu RAG đơn giản, vét cạn.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Ứng dụng kinh doanh của các dự án trong tuần này</h2>
            <span style="color:#181;">RAG có lẽ là kỹ thuật có khả năng ứng dụng tức thì cao nhất trong tất cả những nội dung chúng ta học trong khóa này! Trên thực tế, đã có các sản phẩm thương mại thực hiện chính xác những gì chúng ta xây dựng trong tuần này: truy vấn tinh tế trên các cơ sở dữ liệu thông tin lớn, chẳng hạn như hợp đồng của công ty hoặc thông số kỹ thuật sản phẩm. RAG cung cấp cho bạn một cơ chế có chi phí thấp, nhanh chóng đưa ra thị trường để điều chỉnh LLM phù hợp với lĩnh vực kinh doanh của mình.</span>
        </td>
    </tr>
</table>

In [ ]:
import os  # Nạp os để sử dụng trong notebook.
import glob  # Nạp glob để sử dụng trong notebook.
from dotenv import load_dotenv  # Nhập load_dotenv từ gói dotenv.
from pathlib import Path  # Nhập Path từ gói pathlib.
import gradio as gr  # Nạp gradio as gr để sử dụng trong notebook.
from openai import OpenAI  # Nhập OpenAI từ gói openai.

In [ ]:
# Thiết lập

load_dotenv(override=True)  # Nạp lại các biến trong tệp `.env` vào môi trường chạy.
openai_api_key = os.getenv('OPENAI_API_KEY')  # Đọc khóa API OpenAI từ biến môi trường.
if openai_api_key:  # Kiểm tra khóa API OpenAI đã được cấu hình hay chưa.
    print(f"Khóa API OpenAI tồn tại và bắt đầu bằng {openai_api_key[:8]}")  # Thông báo đã tìm thấy khóa API mà không in toàn bộ khóa.
else:  # Xử lý trường hợp điều kiện phía trên không đúng.
    print("Chưa thiết lập khóa API OpenAI")  # In giá trị hoặc thông báo này ra phần output của ô.

MODEL = "gpt-4.1-nano"  # Chọn tên mô hình ngôn ngữ sẽ được sử dụng.
openai = OpenAI()  # Khởi tạo client OpenAI để gọi các API.

### Hãy đọc toàn bộ dữ liệu nhân viên vào một từ điển

In [ ]:
knowledge = {}  # Khởi tạo từ điển dùng làm cơ sở tri thức trong bộ nhớ.

filenames = glob.glob("knowledge-base/employees/*")  # Lấy danh sách đường dẫn tệp khớp với mẫu glob.

for filename in filenames:  # Lặp qua từng tệp đã tìm thấy.
    name = Path(filename).stem.split(' ')[-1]  # Rút tên mục tri thức từ tên tệp hiện tại.
    with open(filename, "r", encoding="utf-8") as f:  # Mở tệp hiện tại ở chế độ đọc văn bản UTF-8 và tự động đóng sau khi dùng.
        knowledge[name.lower()] = f.read()  # Đọc nội dung tệp và lưu vào từ điển theo khóa viết thường.

In [ ]:
knowledge  # Hiển thị giá trị này trong output của notebook.

In [ ]:
knowledge["lancaster"]  # Hiển thị giá trị này trong output của notebook.

In [ ]:
filenames = glob.glob("knowledge-base/products/*")  # Lấy danh sách đường dẫn tệp khớp với mẫu glob.

for filename in filenames:  # Lặp qua từng tệp đã tìm thấy.
    name = Path(filename).stem  # Rút tên mục tri thức từ tên tệp hiện tại.
    with open(filename, "r", encoding="utf-8") as f:  # Mở tệp hiện tại ở chế độ đọc văn bản UTF-8 và tự động đóng sau khi dùng.
        knowledge[name.lower()] = f.read()  # Đọc nội dung tệp và lưu vào từ điển theo khóa viết thường.

In [ ]:
knowledge.keys()  # Hiển thị toàn bộ khóa hiện có trong từ điển cơ sở tri thức.

In [ ]:
SYSTEM_PREFIX = """
Bạn đại diện cho Insurellm, công ty công nghệ bảo hiểm.
Bạn là chuyên gia trả lời các câu hỏi về Insurellm, nhân viên và sản phẩm của công ty.
Bạn được cung cấp ngữ cảnh bổ sung có thể liên quan đến câu hỏi của người dùng.
Hãy trả lời ngắn gọn và chính xác. Nếu không biết câu trả lời, hãy nói rõ điều đó.

Ngữ cảnh liên quan:
"""  # Hoàn tất nội dung chuỗi nhiều dòng cho `SYSTEM_PREFIX`.

In [ ]:
def get_relevant_context_simple(message):  # Khai báo hàm tìm ngữ cảnh liên quan bằng cách duyệt từng từ trong câu hỏi.
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())  # Lọc câu hỏi để chỉ giữ chữ cái và khoảng trắng.
    words = text.lower().split()  # Chuẩn hóa văn bản thành danh sách từ viết thường.
    relevant_context = []  # Khởi tạo danh sách chứa các phần ngữ cảnh liên quan.
    for word in words:  # Lặp qua từng phần tử của tập dữ liệu này.
        if word in knowledge:  # Chỉ chạy khối lệnh sau khi điều kiện này đúng.
            relevant_context.append(knowledge[word])  # Thêm mục tri thức khớp vào danh sách ngữ cảnh.
    return relevant_context  # Trả về danh sách ngữ cảnh liên quan đã tìm được.

## Nhưng đây là một cách viết đậm chất Python hơn:

In [ ]:
def get_relevant_context(message):  # Khai báo hàm rút gọn để tìm các mục tri thức khớp với từ trong câu hỏi.
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())  # Lọc câu hỏi để chỉ giữ chữ cái và khoảng trắng.
    words = text.lower().split()  # Chuẩn hóa văn bản thành danh sách từ viết thường.
    return [knowledge[word] for word in words if word in knowledge]  # Trả kết quả này về cho nơi gọi hàm.

In [ ]:
get_relevant_context("Lancaster là ai?")  # Thử truy xuất ngữ cảnh liên quan cho câu hỏi mẫu và hiển thị kết quả.

In [ ]:
get_relevant_context("Lancaster là ai và carllm là gì?")  # Thử truy xuất ngữ cảnh liên quan cho câu hỏi mẫu và hiển thị kết quả.

In [ ]:
def additional_context(message):  # Khai báo hàm định dạng phần ngữ cảnh bổ sung cho prompt hệ thống.
    relevant_context = get_relevant_context(message)  # Khởi tạo danh sách chứa các phần ngữ cảnh liên quan.
    if not relevant_context:  # Kiểm tra không tìm thấy ngữ cảnh bổ sung nào.
        result = "Không có ngữ cảnh bổ sung nào liên quan đến câu hỏi của người dùng."  # Lưu kết quả của bước xử lý hiện tại.
    else:  # Xử lý trường hợp điều kiện phía trên không đúng.
        result = "Ngữ cảnh bổ sung sau đây có thể hữu ích khi trả lời câu hỏi của người dùng:\n\n"  # Lưu kết quả của bước xử lý hiện tại.
        result += "\n\n".join(relevant_context)  # Nối các phần ngữ cảnh tìm được vào thông báo kết quả.
    return result  # Trả kết quả này về cho nơi gọi hàm.

In [ ]:
print(additional_context("Alex Lancaster là ai?"))  # In giá trị hoặc thông báo này ra phần output của ô.

In [ ]:
def chat(message, history):  # Khai báo hàm xử lý một lượt trò chuyện và trả về câu trả lời của mô hình.
    system_message = SYSTEM_PREFIX + additional_context(message)  # Ghép prompt hệ thống với ngữ cảnh được truy xuất.
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]  # Tạo danh sách tin nhắn theo định dạng mà mô hình yêu cầu.
    response = openai.chat.completions.create(model=MODEL, messages=messages)  # Gọi mô hình và lưu phản hồi trả về.
    return response.choices[0].message.content  # Trả về nội dung câu trả lời đầu tiên của mô hình.

## Bây giờ chúng ta sẽ đưa ứng dụng này lên Gradio bằng giao diện Chat

Một cách nhanh chóng và dễ dàng để tạo nguyên mẫu trò chuyện với LLM

In [ ]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)  # Khởi chạy giao diện Gradio để người dùng trò chuyện với hệ thống.